# Parameter Golf - T4 Smoke Test
Validates the full training pipeline on a free T4 GPU.
SP8192 config, MLP_MULT=3, int6+brotli compression.

In [ ]:
# Cell 1: Install dependencies
!pip install -q sentencepiece brotli zstandard

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# Cell 2: Clone repo
!rm -rf parameter-golf
!git clone https://github.com/kailean/parameter-golf.git
%cd parameter-golf

In [ ]:
# Cell 3: Copy data from Kaggle dataset
import os, shutil, glob

# Kaggle dataset mount point
dataset_path = '/kaggle/input/parameter-golf-data'
if not os.path.exists(dataset_path):
    # Fallback: download from HF if available
    dataset_path = None

os.makedirs('data/tokenizers', exist_ok=True)
os.makedirs('data/datasets/fineweb10B_sp8192', exist_ok=True)

if dataset_path and os.path.exists(dataset_path):
    # Copy tokenizers
    for f in glob.glob(f'{dataset_path}/data/tokenizers/*.model'):
        shutil.copy2(f, 'data/tokenizers/')
        print(f'Copied {os.path.basename(f)}')
    # Copy data
    for f in glob.glob(f'{dataset_path}/data/datasets/fineweb10B_sp8192/*.bin'):
        shutil.copy2(f, 'data/datasets/fineweb10B_sp8192/')
        print(f'Copied {os.path.basename(f)}')
    print(f'Data ready: {os.listdir("data/tokenizers")}, {os.listdir("data/datasets/fineweb10B_sp8192")}')
else:
    print('ERROR: No Kaggle dataset found! Upload data as a Kaggle dataset first.')

!ls -la data/tokenizers/ data/datasets/fineweb10B_sp8192/

In [ ]:
# Cell 4: Quick pipeline validation
import sys, time, io
sys.path.insert(0, '.')
import torch
import numpy as np
import brotli
from train_gpt_kl import Hyperparameters, GPT, quantize_state_dict_int6, dequantize_state_dict_int6

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
args = Hyperparameters()
print(f'Config: vocab={args.vocab_size}, layers={args.num_layers}, dim={args.model_dim}')

model = GPT(
    vocab_size=args.vocab_size, num_layers=args.num_layers, model_dim=args.model_dim,
    num_heads=args.num_heads, num_kv_heads=args.num_kv_heads, mlp_mult=args.mlp_mult,
    tie_embeddings=args.tie_embeddings, tied_embed_init_std=args.tied_embed_init_std,
    logit_softcap=args.logit_softcap, rope_base=args.rope_base,
    qk_gain_init=args.qk_gain_init, bigram_hash_size=args.bigram_hash_size,
    use_ortho_init=args.use_ortho_init, smear_enabled=args.smear_enabled,
    xsa_last_n=args.xsa_last_n, rope_dims=args.rope_dims,
    ln_scale_enabled=args.ln_scale_enabled
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} params ({n_params/1e6:.1f}M)')

x = torch.randint(0, args.vocab_size, (2, 128), device=device)
y = torch.randint(0, args.vocab_size, (2, 128), device=device)
with torch.no_grad():
    loss = model(x, y)
print(f'Forward pass OK: loss = {loss.item():.4f}')

state_dict = model.state_dict()
quantized = quantize_state_dict_int6(state_dict, use_gptq_lite=args.use_gptq_lite)
buf = io.BytesIO()
torch.save(quantized, buf)
raw_bytes = buf.getvalue()
compressed = brotli.compress(raw_bytes)
mb = len(compressed) / 1024 / 1024
print(f'Int6+Brotli: {len(raw_bytes):,} -> {len(compressed):,} bytes ({mb:.2f} MB)')
print(f'Under 16MB: {"YES" if len(compressed) < 16*1024*1024 else "NO"}')

decompressed = brotli.decompress(compressed)
quant_rt = torch.load(io.BytesIO(decompressed), map_location='cpu', weights_only=False)
dequant = dequantize_state_dict_int6(quant_rt)
max_diff = max((state_dict[k].float() - dequant[k].float()).abs().max().item() for k in state_dict if k in dequant)
print(f'Round-trip max diff: {max_diff:.6f}')
print('ALL PIPELINE CHECKS PASSED')

In [ ]:
# Cell 5: Run full training (SP8192, MLP_MULT=3)
import os
os.environ['DATA_PATH'] = './data/datasets/fineweb10B_sp8192'
os.environ['TOKENIZER_PATH'] = './data/tokenizers/fineweb_8192_bpe.model'
os.environ['VOCAB_SIZE'] = '8192'
os.environ['NUM_LAYERS'] = '11'
os.environ['MODEL_DIM'] = '512'
os.environ['NUM_HEADS'] = '8'
os.environ['NUM_KV_HEADS'] = '4'
os.environ['MLP_MULT'] = '3'
os.environ['TIE_EMBEDDINGS'] = '1'
os.environ['MAX_WALLCLOCK_SECONDS'] = '3600'
os.environ['ITERATIONS'] = '2000'
os.environ['WORLD_SIZE'] = '1'

!python3 train_gpt_kl.py 2>&1 | tee training_output.log

print('\n=== Training complete ===')
!ls -la final_model* submission* *.ptz 2>/dev/null || echo 'Checking for output files...'